# SAIA2163 Final Project - Toxic Comment Detector

**Dataset:** Jigsaw Toxic Comment Classification Challenge  
**Task:** Binary classification - toxic vs non-toxic  
**Checkpoint 1:** Data loading, preprocessing pipeline, baseline model

## 1. Setup & Imports

In [4]:
import nltk, string, regex
import polars as pl
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize, sent_tokenize, RegexpTokenizer
from nltk.corpus import stopwords as nltk_stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from collections import Counter
from langdetect import detect, LangDetectException
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [5]:
nltk.download("stopwords")
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("omw-1.4", quiet=True)


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/kirakuri/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2. Data Loading & Exploration

In [6]:
df_en = pl.read_csv("/home/kirakuri/final-project-nlp/Data/train.csv").select(["comment_text", "toxic"])
df_en = df_en.with_columns([
    pl.lit("en").alias("lang"),
    pl.lit("raw").alias("clean_status"),
])

# Malay file — already has comment_text/toxic/lang
df_ms = pl.read_csv("/home/kirakuri/final-project-nlp/Data/Malay Local Reformat.csv")
df_ms = df_ms.with_columns(pl.lit("cleaned").alias("clean_status"))

Dataset = pl.concat([df_en, df_ms], how="vertical")
print(Dataset.group_by(["lang", "clean_status"]).len())
print(Dataset.shape)
print(Dataset.head(3))


shape: (3, 3)
┌──────┬──────────────┬────────┐
│ lang ┆ clean_status ┆ len    │
│ ---  ┆ ---          ┆ ---    │
│ str  ┆ str          ┆ u32    │
╞══════╪══════════════╪════════╡
│ ms   ┆ cleaned      ┆ 13376  │
│ en   ┆ raw          ┆ 159571 │
│ en   ┆ cleaned      ┆ 13609  │
└──────┴──────────────┴────────┘
(186556, 4)
shape: (3, 4)
┌─────────────────────────────────┬───────┬──────┬──────────────┐
│ comment_text                    ┆ toxic ┆ lang ┆ clean_status │
│ ---                             ┆ ---   ┆ ---  ┆ ---          │
│ str                             ┆ i64   ┆ str  ┆ str          │
╞═════════════════════════════════╪═══════╪══════╪══════════════╡
│ Explanation                     ┆ 0     ┆ en   ┆ raw          │
│ Why the edits made…             ┆       ┆      ┆              │
│ D'aww! He matches this backgro… ┆ 0     ┆ en   ┆ raw          │
│ Hey man, I'm really not trying… ┆ 0     ┆ en   ┆ raw          │
└─────────────────────────────────┴───────┴──────┴──────────────┘


In [7]:
info_test=pl.DataFrame({
    "Column": Dataset.columns,
    "Dtype": [str(dtype) for dtype in Dataset.dtypes],
    "Non-Null Count": [len(Dataset) - count for count in Dataset.null_count().to_numpy()[0]]
})
print(info_test)
Dataset["comment_text"].str.len_chars().describe()


shape: (4, 3)
┌──────────────┬────────┬────────────────┐
│ Column       ┆ Dtype  ┆ Non-Null Count │
│ ---          ┆ ---    ┆ ---            │
│ str          ┆ str    ┆ u32            │
╞══════════════╪════════╪════════════════╡
│ comment_text ┆ String ┆ 186556         │
│ toxic        ┆ Int64  ┆ 186556         │
│ lang         ┆ String ┆ 186556         │
│ clean_status ┆ String ┆ 186556         │
└──────────────┴────────┴────────────────┘


statistic,value
str,f64
"""count""",186556.0
"""null_count""",0.0
"""mean""",354.770884
"""std""",555.347235
"""min""",6.0
"""25%""",87.0
"""50%""",180.0
"""75%""",376.0
"""max""",5000.0


In [8]:
Dataset["toxic"].value_counts()

toxic,count
i64,u32
1,27637
0,158919


## 3. Preprocessing Pipeline

### 3.1 Remove Null & Empty Rows

In [9]:
# Extract as list for NLP
toxic_corpus = list(zip(
    Dataset["comment_text"].to_list(),
    Dataset["toxic"].to_list()
))

In [10]:
# Save the non-whitespace
Dataset = Dataset.filter(
    pl.col("comment_text").is_not_null() &
    (pl.col("comment_text").str.strip_chars() != "")
)

Dataset[["comment_text","toxic"]].write_csv("/home/kirakuri/final-project-nlp/Data/toxic_corpus.csv")
print(f"Saved toxic_corpus.csv: {Dataset.shape[0]} rows")


Saved toxic_corpus.csv: 186556 rows


In [11]:
print("After null filter:", Dataset.shape)

After null filter: (186556, 4)


### 3.2 Language Detection

*Retained for documentation. Language filtering skipped since Jigsaw is ~98% English. Non-English rows retained as noise impact is negligible.*

In [12]:
def safe_detect(text):
    try:
        if not text or text.strip() == "":
            return "unknown"
        return detect(text)
    except LangDetectException:
        return "unknown"
    except Exception:
        return "unknown"

"""
Dataset = Dataset.with_columns(
    pl.col("comment_text").map_elements(
        safe_detect,
        return_dtype=pl.String
    ).alias("language")
)
print(Dataset["language"].value_counts().sort("count", descending=True))
"""

'\nDataset = Dataset.with_columns(\n    pl.col("comment_text").map_elements(\n        safe_detect,\n        return_dtype=pl.String\n    ).alias("language")\n)\nprint(Dataset["language"].value_counts().sort("count", descending=True))\n'

In [13]:
"""
# Keep only English rows
Dataset = Dataset.filter(
    pl.col("language") == "en"
)

print(Dataset.shape)
print(Dataset["language"].value_counts())
"""

'\n# Keep only English rows\nDataset = Dataset.filter(\n    pl.col("language") == "en"\n)\n\nprint(Dataset.shape)\nprint(Dataset["language"].value_counts())\n'

### 3.3 Text Cleaning

Removes URLs, HTML/wiki markup, hex colors, CSS properties, keyboard spam, and special characters.

In [14]:
def clean_text(text):
    if not text or text.strip() == "":
        return ""

    # 1. Remove URLs
    text = regex.sub(r'https?://\S+', '', text)

    # 2. Remove HTML/Wiki markup
    text = regex.sub(r'\{\|.*?\|\}', '', text, flags=regex.DOTALL)  # wiki tables
    text = regex.sub(r'\|[\w\s=""""#;:]+', '', text)                # wiki style attrs
    text = regex.sub(r'style="""".*?""""', '', text)                 # style attributes
    text = regex.sub(r'<.*?>', '', text)                             # HTML tags

    # 3. Remove hex colors
    text = regex.sub(r'#[0-9a-fA-F]{3,6}', '', text)

    # 4. Remove CSS properties
    text = regex.sub(r'\b(background-color|vertical-align|padding|border|font-size|rowspan|colspan|height|width)\b.*?;', '', text)

    # 5. Remove keyboard spam — words longer than 25 chars with no vowel pattern
    words = text.split()
    words = [w for w in words if not (len(w) > 25 and regex.search(r'[^aeiou]{10,}', w.lower()))]
    text = " ".join(words)

    # 6. Remove pure numbers and number-heavy tokens
    text = regex.sub(r'\b\d+[\d.,]+\b', '', text)

    # 7. Remove leftover symbols
    text = regex.sub(r'[^\w\s]', ' ', text)

    # 8. Remove extra whitespace
    text = regex.sub(r'\s+', ' ', text).strip()

    return text


In [15]:
def maybe_clean(args):
    text, status = args
    # For Malay dataset
    return clean_text(text) if status == "raw" else text

In [16]:
rows_for_cleaning = list(zip(
    Dataset["comment_text"].to_list(),
    Dataset["clean_status"].to_list()
))

with Pool(cpu_count()) as pool:
    cleaned_results = list(tqdm(
        pool.imap(maybe_clean, rows_for_cleaning, chunksize=500),
        total=len(rows_for_cleaning),
        desc="Cleaning"
    ))

Dataset = Dataset.with_columns(
    pl.Series("cleaned_text", cleaned_results, dtype=pl.String)
)

# Remove empty after cleaning
Dataset = Dataset.filter(
    pl.col("cleaned_text").is_not_null() &
    (pl.col("cleaned_text").str.strip_chars() != "")
)
print(f"After cleaning: {Dataset.shape[0]} rows")


Cleaning: 100%|██████████| 186556/186556 [00:01<00:00, 129754.54it/s]


After cleaning: 186385 rows


### 3.4 Slang Detection

Identifies unusual tokens for manual review and slang map construction.

In [17]:
def slang_word(text):
    try:
        unusual = []
        words = regex.findall(r'\b\w+\b', text.lower())

        for word in words:
            if len(word) < 3:
                continue
            if (
                regex.search(r'(.)\1{1,}', word) or
                regex.search(r'[a-z]+[0-9]+[a-z]*', word) or
                regex.search(r'[0-9]+[a-z]+', word) or
                regex.search(r'(.)\1{2,}', word)
            ):
                unusual.append(word)

        return ", ".join(unusual) if unusual else ""
    except:
        return ""

In [18]:
texts = Dataset["cleaned_text"].to_list()

with Pool(cpu_count()) as pool:
    results = list(tqdm(
        pool.imap(slang_word, texts),
        total=len(texts)
    ))

Dataset = Dataset.with_columns(
    pl.Series("unusual_words", results)
)

review_df = Dataset.filter(
    pl.col("unusual_words") != ""
)[["comment_text", "unusual_words","toxic"]]

print(review_df.head(10))

100%|██████████| 186385/186385 [00:18<00:00, 10089.99it/s]


shape: (10, 3)
┌─────────────────────────────────┬─────────────────────────────────┬───────┐
│ comment_text                    ┆ unusual_words                   ┆ toxic │
│ ---                             ┆ ---                             ┆ ---   │
│ str                             ┆ str                             ┆ i64   │
╞═════════════════════════════════╪═════════════════════════════════╪═══════╡
│ Explanation                     ┆ metallica, dolls                ┆ 0     │
│ Why the edits made…             ┆                                 ┆       │
│ D'aww! He matches this backgro… ┆ aww, seemingly                  ┆ 0     │
│ Hey man, I'm really not trying… ┆ really, seems, formatting       ┆ 0     │
│ "                               ┆ suggestions, accidents, need, … ┆ 0     │
│ More                            ┆                                 ┆       │
│ I can't make any real s…        ┆                                 ┆       │
│ "                               ┆ well, tools, 

In [19]:
# Saving slang word in csv
review_df.write_csv("/home/kirakuri/final-project-nlp/Data/slang_review.csv")
print(f"Rows to review: {review_df.shape[0]}")


Rows to review: 156813


### 3.5 Slang Normalization

Maps obfuscated toxic words to canonical forms before tokenization.

In [20]:
slang_map = {
    # faggot variants
    'faggt': 'faggot',
    'faggo': 'faggot',
    'fagget': 'faggot',
    'faggit': 'faggot',
    'fagg': 'faggot',
    'faggets': 'faggot',
    'faggoty': 'faggot',
    'faggish': 'faggot',
    # fuck variants
    'fukk': 'fuck',
    'fukken': 'fuck',
    'fukking': 'fuck',
    'fukker': 'fuck',
    'fukkin': 'fuck',
    'fukked': 'fuck',
    'fuuck': 'fuck',
    'fuucker': 'fuck',
    'fuckk': 'fuck',
    'fuckoff': 'fuck',
    'fuckking': 'fuck',
    # shit variants
    'shiit': 'shit',
    'shitt': 'shit',
    'shitty': 'shit',
    'shittin': 'shit',
    'shitter': 'shit',
    # nigger variants
    'niggaz': 'nigger',
    'niggah': 'nigger',
    'niggars': 'nigger',
    'nigga': 'nigger',
    'niggas': 'nigger',
    'niggerlover': 'nigger',
    'niggering': 'nigger',
    'niggerism': 'nigger',
    'niggerfaggot': 'nigger',
    'sandnigger': 'nigger',
    'sandniggers': 'nigger',
    # compound ass words
    'dumbass': 'dumb',
    'jackass': 'jerk',
    'asshole': 'asshole',
    'assholes': 'asshole',
    'asshat': 'asshole',
    'asswipe': 'asshole',
    'assclown': 'asshole',
    'assbitch': 'asshole',
    'bitchass': 'asshole',
    'cuntass': 'asshole',
    'punkass': 'asshole',
    'lameass': 'asshole',
    'gayass': 'asshole',
    'fatass': 'asshole',
    'smartass': 'asshole',
    'badass': 'asshole',
    # pussy variants
    'pussyfuck': 'pussy',
    'pussylicker': 'pussy',
    'pussyass': 'pussy',
    # misc
    'goddamned': 'goddamn',
    'goddamnit': 'goddamn',
    'bullshitter': 'bullshit',
    'bullshitting': 'bullshit',
    'pissfuck': 'fuck',
    'buttfuck': 'fuck',
    'killyou': 'kill',
}

In [21]:
malayslangdict = {
    "2": "itu",
    "6be": "nombor",
    "abeh": "habis",
    "abes": "habis",
    "abih": "habis",
    "abg": "abang",
    "ade": "ada",
    "ader": "ada",
    "ado": "ada",
    "diaorg": "dia orang", 
    "xtau": "tak tahu",
    "xde": "takde", 
    "xnak": "tak nak", 
    "siallah": "sial lah",
    "ape": "apa", 
    "skrg": "sekarang", 
    "wtf": "what the heck",
    "fucuk": "fuck", 
    "cb": "cibai", 
    "knl": "kenal",
    "bodohla": "bodoh la",
    "hado": "ada",
    "adk": "adik",
    "adek": "adik",
    "adeq": "adik",
    "adlh": "adalah",
    "agk": "agak",
    "aje": "sahaja",
    "ajer": "sahaja",
    "je": "sahaja",
    "shj": "sahaja",
    "ja": "sahaja",
    "jek": "sahaja",
    "jer": "sahaja",
    "jerk": "sahaja",
    "saje": "sahaja",
    "ak": "aku",
    "ako": "aku",
    "aq": "aku",
    "akk": "kakak",
    "akn": "akan",
    "ambik": "ambil",
    "amek": "ambil",
    "hambek": "ambil",
    "ank": "anak",
    "antar": "hantar",
    "ap": "apa",
    "aper": "apa",
    "hapa": "apa",
    "haper": "apa",
    "apasal": "apa pasal",
    "apesal": "apa pasal",
    "apahal": "apa hal",
    "apehal": "apa hal",
    "apetah": "apa entah",
    "arap": "harap",
    "ari": "hari",
    "aritu": "hari itu",
    "asalkn": "asalkan",
    "aslkn": "asalkan",
    "asik": "asyik",
    "asl": "asal",
    "ati": "hati",
    "ats": "atas",
    "awat": "kenapa",
    "awt": "kenapa",
    "awk": "awak",
    "awl": "awal",
    "ayer": "air",
    "ayh": "ayah",
    "ayt": "ayat",
    "b": "ber",
    "br": "ber",
    "bagitau": "bagi tahu",
    "bgitau": "bagi tahu",
    "bgtu": "bagi tahu",
    "bgtaw": "bagi tahu",
    "bitau": "bagi tahu",
    "bapak": "bapa",
    "bape": "berapa",
    "baper": "berapa",
    "brp": "berapa",
    "bpe": "berapa",
    "braper": "berapa",
    "bby": "baby",
    "bc": "baca",
    "bca": "baca",
    "bdak": "budak",
    "dak": "budak",
    "bdn": "badan",
    "brd": "bandar",
    "bndar": "bandar",
    "bdoh": "bodoh",
    "beb": "babe",
    "beger": "burger",
    "bekfes": "breakfast",
    "belen": "balance",
    "beno": "benar",
    "benor": "benar",
    "bnr": "benar",
    "bbeno": "benar-benar",
    "bebeno": "benar-benar",
    "bebenor": "benar-benar",
    "benti": "berhenti",
    "bes": "best",
    "besaq": "besar",
    "beso": "besar",
    "besor": "besar",
    "bese": "biasa",
    "biase": "biasa",
    "betoi": "betul",
    "betol": "betul",
    "betui": "betul",
    "bg": "bagi",
    "bgi": "bagi",
    "bgai": "bagai",
    "bgini": "begini",
    "gini": "begini",
    "bgitu": "begitu",
    "gitu": "begitu",
    "bgkai": "bangkai",
    "bgkus": "bungkus",
    "bgn": "bangun",
    "bgs": "bagus",
    "bgus": "bagus",
    "bgsa": "bangsa",
    "bgun": "bangun",
    "bhasa": "bahasa",
    "bhs": "bahasa",
    "bhse": "bahasa",
    "bhgn": "bahagian",
    "bia": "biar",
    "biaq": "biar",
    "bile": "bila",
    "bla": "bila",
    "ble": "bila",
    "bior": "biar",
    "bj": "baju",
    "bju": "baju",
    "bjet": "bajet",
    "bjya": "berjaya",
    "bk": "buka",
    "bka": "buka",
    "bkak": "buka",
    "bkk": "buka",
    "bkn": "bukan",
    "blah": "pergi",
    "blaja": "belajar",
    "blja": "belajar",
    "bljar": "belajar",
    "bljr": "belajar",
    "blanja": "belanja",
    "blnj": "belanja",
    "bleh": "boleh",
    "blh": "boleh",
    "bley": "boleh",
    "blek": "balik",
    "blik": "balik",
    "blk": "balik",
    "blkg": "belakang",
    "blkng": "belakang",
    "blm": "belum",
    "blom": "belum",
    "blum": "belum",
    "bln": "bulan",
    "blj": "belanja",
    "blnja": "belanja",
    "bls": "balas",
    "bnd": "benda",
    "bnde": "benda",
    "bndg": "banding",
    "bngsa": "bangsa",
    "bntu": "bantu",
    "bnyk": "banyak",
    "byk": "banyak",
    "bodo": "bodoh",
    "borg": "borang",
    "borng": "borang",
    "br": "baru",
    "brader": "bro",
    "brg": "barang",
    "brlku": "berlaku",
    "bru": "baru",
    "bsr": "besar",
    "bsar": "besar",
    "bt": "buat",
    "bwat": "buat",
    "wat": "buat",
    "wt": "buat",
    "btl": "betul",
    "btpe": "buat apa",
    "watpe": "buat apa",
    "wtpe": "buat apa",
    "bw": "bawa",
    "bwa": "bawa",
    "bwak": "bawa",
    "bwh": "bawah",
    "byk": "banyak",
    "byak": "banyak",
    "byr": "bayar",
    "camna": "macam mana",
    "camne": "macam mana",
    "camner": "macam mana",
    "cmna": "macam mana",
    "cmne": "macam mana",
    "cmner": "macam mana",
    "cmana": "macam mana",
    "cmane": "macam mana",
    "cmaner": "macam mana",
    "camni": "macam ni",
    "cmnie": "macam ni",
    "cbe": "cuba",
    "ce": "cuba",
    "cer": "cuba",
    "cbok": "sibuk",
    "cdgn": "cadangan",
    "cdgan": "cadangan",
    "cdgkn": "cadangkan",
    "cdgkan": "cadangkan",
    "chat": "sihat",
    "cite": "cerita",
    "citer": "cerita",
    "crita": "cerita",
    "crite": "cerita",
    "crta": "cerita",
    "cter": "cerita",
    "cket": "sikit",
    "ckit": "sikit",
    "skit": "sikit",
    "ckp": "cakap",
    "ckap": "cakap",
    "ckup": "cukup",
    "cm": "macam",
    "mcm": "macam",
    "cmburu": "cemburu",
    "cme": "cuma",
    "cmpur": "campur",
    "cmtu": "camtu",
    "cna": "sana",
    "cni": "sini",
    "cnta": "cinta",
    "cntik": "cantik",
    "cntk": "cantik",
    "come": "comel",
    "comei": "comel",
    "comolot": "cium mulut",
    "cpt": "cepat",
    "cepat": "cepat",
    "cr": "cara",
    "cri": "cari",
    "crik": "cari",
    "ct": "siti",
    "cth": "contoh",
    "cntoh": "contoh",
    "ctu": "situ",
    "stu": "situ",
    "d": "di",
    "da": "sudah",
    "dah": "sudah",
    "dh": "sudah",
    "ddk": "duduk",
    "dduk": "duduk",
    "duk": "duduk",
    "de": "ada",
    "dea": "dia",
    "dek": "adik",
    "deq": "adik",
    "dik": "adik",
    "derg": "dia orang",
    "dorg": "dia orang",
    "diaorg": "dia orang",
    "diorg": "dia orang",
    "dorng": "dia orang",
    "dgn": "dengan",
    "dgan": "dengan",
    "dgr": "dengar",
    "dgar": "dengar",
    "die": "dia",
    "dier": "dia",
    "dkt": "dekat",
    "dkat": "dekat",
    "dl": "dahulu",
    "dlu": "dahulu",
    "dlm": "dalam",
    "dlam": "dalam",
    "dn": "dan",
    "dpt": "dapat",
    "dpat": "dapat",
    "dpn": "depan",
    "dpan": "depan",
    "dr": "dari",
    "dari": "dari",
    "drpd": "daripada",
    "dtg": "datang",
    "ekceli": "actually",
    "fhm": "faham",
    "fkir": "fikir",
    "pk": "fikir",
    "fmly": "family",
    "g": "pergi",
    "gado": "gaduh",
    "gdh": "gaduh",
    "gak": "juga",
    "giler": "gila",
    "gle": "gila",
    "gler": "gila",
    "gini": "begini",
    "gitu": "begitu",
    "gtuh": "begitu",
    "gtu": "begitu",
    "gmba": "gambar",
    "gmbar": "gambar",
    "gmbor": "gambar",
    "gmbr": "gambar",
    "gna": "guna",
    "gne": "guna",
    "hbgn": "hubungan",
    "hbs": "habis",
    "hdp": "hadap",
    "hdup": "hidup",
    "hepi": "happy",
    "heran": "hairan",
    "hg": "hang",
    "hgpa": "hangpa",
    "hjg": "hujung",
    "hlg": "hilang",
    "hmpa": "hampa",
    "hmpir": "hampir",
    "hntr": "hantar",
    "hny": "hanya",
    "hnya": "hanya",
    "hnye": "hanya",
    "hosp": "hospital",
    "i": "saya",
    "spital": "hospital",
    "hrga": "harga",
    "hri": "hari",
    "hrp": "harap",
    "hrap": "harap",
    "hutg": "hutang",
    "hutng": "hutang",
    "igt": "ingat",
    "jap": "sekejap",
    "jd": "jadi",
    "jdi": "jadi",
    "jeles": "jealous",
    "jg": "juga",
    "jga": "juga",
    "jgk": "juga",
    "jgak": "juga",
    "jugak": "juga",
    "juge": "juga",
    "jgn": "jangan",
    "jilake": "celaka",
    "jln": "jalan",
    "jmp": "jumpa",
    "jmpa": "jumpa",
    "jnis": "jenis",
    "jnji": "janji",
    "jntn": "jantan",
    "jwb": "jawab",
    "jawab": "jawab",
    "jwpn": "jawapan",
    "jwtn": "jawatan",
    "k": "okay",
    "ka": "ke",
    "kabo": "khabar",
    "kate": "kata",
    "kawen": "kahwin",
    "kawin": "kahwin",
    "kcik": "kecil",
    "kecik": "kecil",
    "kechik": "kecil",
    "kcuali": "kecuali",
    "kdg2": "kadang-kadang",
    "kekadang": "kadang-kadang",
    "keja": "kerja",
    "keje": "kerja",
    "kejer": "kerja",
    "kje": "kerja",
    "krja": "kerja",
    "kg": "kampung",
    "kite": "kita",
    "kte": "kita",
    "kjaan": "kerajaan",
    "krjaan": "kerajaan",
    "klu": "kalau",
    "kn": "kan",
    "knl": "kenal",
    "knp": "kenapa",
    "ko": "kau",
    "korg": "kau orang",
    "kpd": "kepada",
    "krn": "kerana",
    "kat": "dekat",
    "kt": "dekat",
    "ktorg": "kita orang",
    "kuar": "keluar",
    "kwn": "kawan",
    "kwsn": "kawasan",
    "lbh": "lebih",
    "lbih": "lebih",
    "lg": "lagi",
    "lgi": "lagi",
    "lgpun": "lagipun",
    "lgsg": "langsung",
    "lh": "lah",
    "lmbt": "lambat",
    "lme": "lama",
    "lncr": "lancar",
    "lyn": "layan",
    "mane": "mana",
    "mbe": "member",
    "mcm": "macam",
    "mcmne": "macam mana",
    "mcmner": "macam mana",
    "mcmni": "macam ini",
    "mcmnie": "macam ini",
    "mcmtu": "macam itu",
    "mcmtue": "macam itu",
    "men": "main",
    "mende": "benda",
    "mggu": "minggu",
    "mgkin": "mungkin",
    "mgkn": "mungkin",
    "mkn": "makan",
    "mksd": "maksud",
    "mmg": "memang",
    "mmng": "memang",
    "mmpu": "mampu",
    "mn": "mana",
    "mne": "mana",
    "mnx": "minta",
    "mintak": "minta",
    "mtk": "minta",
    "mrk": "mereka",
    "msk": "masuk",
    "mslh": "masalah",
    "msti": "mesti",
    "nak": "hendak",
    "nk": "hendak",
    "ne": "mana",
    "ngan": "dengan",
    "ngn": "dengan",
    "ni": "ini",
    "nie": "ini",
    "nih": "ini",
    "nmpk": "nampak",
    "nmpak": "nampak",
    "nnt": "nanti",
    "nnti": "nanti",
    "t": "nanti",
    "ntah": "entah",
    "tah": "entah",
    "nye": "punya",
    "org": "orang",
    "olh": "oleh",
    "p": "pergi",
    "pd": "pada",
    "pdhal": "padahal",
    "pebenda": "apa benda",
    "pg": "pagi",
    "pgi": "pagi",
    "pggl": "panggil",
    "pjg": "panjang",
    "pnjg": "panjang",
    "pk": "fikir",
    "plak": "pula",
    "plg": "paling",
    "plk": "pelik",
    "pn": "pun",
    "pndai": "pandai",
    "pnh": "penuh",
    "prnah": "pernah",
    "pntg": "penting",
    "pnting": "penting",
    "pnye": "punya",
    "ppuan": "perempuan",
    "psl": "fasal",
    "pstu": "lepas itu",
    "ptg": "petang",
    "ptt": "patut",
    "pyh": "payah",
    "retis": "artis",
    "rkn": "rakan",
    "rkyat": "rakyat",
    "rkyt": "rakyat",
    "rmai": "ramai",
    "rmh": "rumah",
    "rs": "rasa",
    "rsa": "rasa",
    "rse": "rasa",
    "sado": "sasa",
    "sape": "siapa",
    "sbb": "sebab",
    "sbg": "sebagai",
    "sbr": "sabar",
    "scr": "secara",
    "sdar": "sedar",
    "sdp": "sedap",
    "sdr": "sedar",
    "sg": "sungai",
    "sgala": "segala",
    "sggh": "singgah",
    "sggp": "sanggup",
    "snggup": "sanggup",
    "sgguh": "sungguh",
    "sgka": "sangka",
    "sgt": "sangat",
    "shbt": "sahabat",
    "skg": "sekarang",
    "skang": "sekarang",
    "skunk": "sekarang",
    "skrang": "sekarang",
    "skrg": "sekarang",
    "sknx": "sekarang",
    "ske": "suka",
    "suke": "suka",
    "sklh": "sekolah",
    "slh": "salah",
    "sllu": "selalu",
    "slmt": "selamat",
    "slamat": "selamat",
    "sme": "sama",
    "smlm": "semalam",
    "smp": "sampai",
    "smpai": "sampai",
    "smpi": "sampai",
    "sndiri": "sendiri",
    "sne": "sana",
    "sng": "senang",
    "sorg": "seorang",
    "spt": "seperti",
    "spy": "supaya",
    "ssh": "susah",
    "sume": "semua",
    "sy": "saya",
    "syg": "sayang",
    "tau": "tahu",
    "taw": "tahu",
    "tawu": "tahu",
    "tdo": "tidur",
    "tido": "tidur",
    "tggi": "tinggi",
    "tnggi": "tinggi",
    "tggjwb": "tanggungjawab",
    "tggl": "tinggal",
    "tgu": "tunggu",
    "tggu": "tunggu",
    "tgh": "tengah",
    "tgk": "tengok",
    "tgok": "tengok",
    "tngk": "tengok",
    "tgn": "tangan",
    "thn": "tahun",
    "tkt": "tingkat",
    "tkut": "takut",
    "tlg": "tolong",
    "tmbah": "tambah",
    "tmbh": "tambah",
    "tmpat": "tempat",
    "tmpt": "tempat",
    "tntg": "tentang",
    "ttg": "tentang",
    "tp": "tapi",
    "tpi": "tapi",
    "tpt": "tepat",
    "trs": "terus",
    "trus": "terus",
    "ttp": "tetap",
    "ttpi": "tetapi",
    "tu": "itu",
    "tue": "itu",
    "utk": "untuk",
    "uol": "you all",
    "uols": "you all",
    "uolls": "you all",
    "wlu": "walau",
    "wlpn": "walaupun",
    "wlpun": "walaupun",
    "wpun": "walaupun",
    "wslm": "waalaikumsalam",
    "x": "tak",
    "xde": "tak ada",
    "xder": "tak ada",
    "takde": "tak ada",
    "takder": "tak ada",
    "xkn": "tak akan",
    "xkan": "tak akan",
    "takkan": "tak akan",
    "xle": "tak boleh",
    "xleh": "tak boleh",
    "takleh": "tak boleh",
    "xmau": "tak mahu",
    "xnk": "tak nak",
    "xnak": "tak nak",
    "taknak": "tak nak",
    "xpe": "tak apa",
    "xper": "tak apa",
    "takpe": "tak apa",
    "takper": "tak apa",
    "y": "yang",
    "yg": "yang",
    "zmn": "zaman"
}

### 3.6 Tokenization, Stopword Removal, Stemming & Lemmatization

In [22]:
stop_words = set(nltk_stopwords.words("english"))
tokenizer = RegexpTokenizer(r"(?u)\w+")
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


In [23]:
import nltk

def get_wordnet_pos(tag):
    """Map POS tag to WordNet POS for lemmatization."""
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def preprocessing(args):
    corpus, lang = args
    corpus = corpus.lower()
    tokens = tokenizer.tokenize(corpus)
    
    if lang == "ms":
        # Malay slang normalize
        tokens = [malayslangdict.get(t, t) for t in tokens]
        return tokens
    
    tokens = [slang_map.get(token, token) for token in tokens]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    pos_tags = nltk.pos_tag(tokens)
    tokens = [lemmatizer.lemmatize(t, get_wordnet_pos(pos)) for t, pos in pos_tags]
    return tokens


In [25]:
rows_for_preprocessing = list(zip(
    Dataset["cleaned_text"].to_list(),
    Dataset["lang"].to_list()
))

with Pool(cpu_count()) as pool:
    Preprocess_data = list(tqdm(
        pool.imap(preprocessing, rows_for_preprocessing),
        total=len(rows_for_preprocessing),
        desc="Preprocessing"
    ))
print(Preprocess_data[:5])

Preprocessing: 100%|██████████| 186385/186385 [00:41<00:00, 4493.77it/s] 


[['explan', 'edit', 'make', 'usernam', 'hardcor', 'metallica', 'fan', 'revert', 'vandal', 'closur', 'ga', 'vote', 'new', 'york', 'doll', 'fac', 'plea', 'remov', 'templat', 'talk', 'page', 'sinc', 'retir'], ['aww', 'match', 'background', 'colour', 'seemingli', 'stick', 'thank', 'talk', 'januari', 'utc'], ['hey', 'man', 'realli', 'tri', 'edit', 'war', 'guy', 'constantli', 'remov', 'relev', 'inform', 'talk', 'edit', 'instead', 'talk', 'page', 'seem', 'care', 'format', 'actual', 'info'], ['make', 'real', 'suggest', 'improv', 'wonder', 'section', 'statist', 'later', 'subsect', 'type', 'accid', 'think', 'refer', 'may', 'need', 'tidi', 'exact', 'format', 'ie', 'date', 'format', 'etc', 'later', 'one', 'el', 'first', 'prefer', 'format', 'style', 'refer', 'want', 'pleas', 'let', 'know', 'appear', 'backlog', 'articl', 'review', 'guess', 'may', 'delay', 'review', 'turn', 'list', 'relev', 'form', 'eg', 'wikipedia', 'good_article_nomin', 'transport'], ['sir', 'hero', 'chanc', 'rememb', 'page']]


In [26]:
# Join tokens back to string to save as CSV
Dataset = Dataset.with_columns(
    pl.Series("preprocessed", Preprocess_data)
)

Dataset = Dataset.with_columns(
    pl.col("preprocessed").map_elements(
        lambda x: " ".join(x),
        return_dtype=pl.String
    ).alias("preprocessed_text")
)

print(Dataset[["comment_text", "preprocessed_text","toxic"]].head(5))

shape: (5, 3)
┌─────────────────────────────────┬─────────────────────────────────┬───────┐
│ comment_text                    ┆ preprocessed_text               ┆ toxic │
│ ---                             ┆ ---                             ┆ ---   │
│ str                             ┆ str                             ┆ i64   │
╞═════════════════════════════════╪═════════════════════════════════╪═══════╡
│ Explanation                     ┆ explan edit make usernam hardc… ┆ 0     │
│ Why the edits made…             ┆                                 ┆       │
│ D'aww! He matches this backgro… ┆ aww match background colour se… ┆ 0     │
│ Hey man, I'm really not trying… ┆ hey man realli tri edit war gu… ┆ 0     │
│ "                               ┆ make real suggest improv wonde… ┆ 0     │
│ More                            ┆                                 ┆       │
│ I can't make any real s…        ┆                                 ┆       │
│ You, sir, are my hero. Any cha… ┆ sir hero chanc

## 4. Save Preprocessed Data

In [27]:
# Drop nulls before saving
Dataset = Dataset.filter(pl.col("preprocessed_text").is_not_null())

Dataset[["preprocessed_text", "toxic"]].write_csv("/home/kirakuri/final-project-nlp/Data/preprocessed_corpus.csv")
print(f"Saved preprocessed_corpus.csv: {Dataset.shape[0]} rows")
print(Dataset[["preprocessed_text", "toxic"]].head(3))


Saved preprocessed_corpus.csv: 186385 rows
shape: (3, 2)
┌─────────────────────────────────┬───────┐
│ preprocessed_text               ┆ toxic │
│ ---                             ┆ ---   │
│ str                             ┆ i64   │
╞═════════════════════════════════╪═══════╡
│ explan edit make usernam hardc… ┆ 0     │
│ aww match background colour se… ┆ 0     │
│ hey man realli tri edit war gu… ┆ 0     │
└─────────────────────────────────┴───────┘


## 5. Class Balancing

In [28]:
# Class balancing — undersample majority class to 50/50
import polars as pl

df_full = pl.read_csv("/home/kirakuri/final-project-nlp/Data/preprocessed_corpus.csv")
df_full = df_full.filter(pl.col("preprocessed_text").is_not_null())

toxic = df_full.filter(pl.col("toxic") == 1)
clean = df_full.filter(pl.col("toxic") == 0)

clean_sampled = clean.sample(n=toxic.shape[0], seed=42)
balanced = pl.concat([toxic, clean_sampled]).sample(fraction=1.0, shuffle=True, seed=42)

balanced.write_csv("/home/kirakuri/final-project-nlp/Data/balanced_corpus.csv")
print(f"Balanced dataset: {balanced.shape}")
print(balanced["toxic"].value_counts())

Balanced dataset: (55262, 2)
shape: (2, 2)
┌───────┬───────┐
│ toxic ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 1     ┆ 27631 │
│ 0     ┆ 27631 │
└───────┴───────┘


## 6. Baseline Model. Logistic Regression + TF-IDF

Trained on balanced dataset (15,288 toxic / 15,288 clean). Evaluated with accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
# Load balanced dataset
df = pd.read_csv("/home/kirakuri/final-project-nlp/Data/balanced_corpus.csv")
df = df.dropna(subset=['preprocessed_text'])

texts_final = df["preprocessed_text"].tolist()
labels = df["toxic"].tolist()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    texts_final, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# Vectorize + Train
tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=["Non-Toxic", "Toxic"])) 
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8892
              precision    recall  f1-score   support

   Non-Toxic       0.88      0.91      0.89      5523
       Toxic       0.90      0.87      0.89      5526

    accuracy                           0.89     11049
   macro avg       0.89      0.89      0.89     11049
weighted avg       0.89      0.89      0.89     11049

[[5004  519]
 [ 705 4821]]


In [ ]:
import pandas as pd
df = pd.read_csv("/home/kirakuri/final-project-nlp/Data/balanced_corpus.csv")
df = df.dropna(subset=['preprocessed_text'])
print(df.shape)
print(df['toxic'].value_counts()) 

(55245, 2)
toxic
1    27629
0    27616
Name: count, dtype: int64
